# Armenian Participle Corpus: Stanza Preprocessing & Balanced Sampling

**Purpose:** Processes multi-million sentence corpora through the Stanza dependency parser to extract token-level morphological and syntactic metadata. Filters the resulting parsed sentences based on quality heuristics, additive counts, and structural complexity to produce a balanced 120K training subset (50K `-լով`, 50K `-ած`, 20K both).

**Inputs:**
- Categorized raw text corpora (`corpus_lov.txt`, `corpus_ac.txt`, `corpus_both.txt`).

**Outputs:**
- Intermediate Stanza metadata shards (`stanza_output/`).
- Final balanced subsets in JSON, JSONL, and TXT formats (`filtered_corpus/`).

**Hardware Requirements:** GPU (e.g., NVIDIA GTX 1070 Ti or better) with at least 8GB VRAM is highly recommended to handle Stanza batch processing efficiently.

**Estimated Runtime:** ~15-25 hours for full corpus parsing on GPU, plus ~10 minutes for filtering and sampling.

## Setup and Configuration

In [ ]:
import os
import re
import json
import time
import random
import logging
from pathlib import Path
from collections import defaultdict
from typing import List, Dict, Optional, Tuple

import torch
import numpy as np
import stanza
from tqdm.notebook import tqdm

torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])
logging.getLogger("stanza").setLevel(logging.WARNING)

DATA_DIR = Path("./data")
INPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "stanza_output"
CHECKPOINT_DIR = DATA_DIR / "checkpoints"
FILTERED_DIR = DATA_DIR / "filtered_corpus"

for d in [OUTPUT_DIR, CHECKPOINT_DIR, FILTERED_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CORPUS_FILES = {
    "lov": INPUT_DIR / "corpus_lov.txt",
    "ac":  INPUT_DIR / "corpus_ac.txt",
    "both": INPUT_DIR / "corpus_both.txt",
}

BATCH_SIZE = 200
CHECKPOINT_EVERY = 5000
USE_GPU = torch.cuda.is_available()

TARGET_LOV = 50_000
TARGET_AC = 50_000
TARGET_BOTH = 20_000

LOV_SUFFIX = "\u056c\u0578\u057e" 
AC_SUFFIX = "\u0561\u056e"        

INDEX_PATTERN = re.compile(r"^\s*\d+\.\s*", re.UNICODE)
ADDITIVE_DEPRELS = {"obj", "iobj", "obl", "advmod", "acl", "ccomp", "nsubj:pass"}
COORD_DEPRELS = {"cc", "conj"}

## Stanza Pipeline and Extraction Logic

In [ ]:
stanza.download("hy", processors="tokenize,pos,lemma,depparse", verbose=False)
nlp = stanza.Pipeline(
    "hy",
    processors="tokenize,pos,lemma,depparse",
    use_gpu=USE_GPU,
    verbose=False,
    tokenize_batch_size=64
)

def read_corpus_file(filepath: Path, max_lines: Optional[int] = None) -> List[Dict]:
    """Reads a corpus file with indexed sentences and strips the index prefix.

    Args:
        filepath (Path): Path to the text corpus.
        max_lines (int, optional): Maximum number of lines to read.

    Returns:
        List[Dict]: List of dictionaries containing original indices and cleaned text.
    """
    sentences = []
    if not filepath.exists():
        return sentences
        
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if max_lines and i >= max_lines:
                break
            line = line.strip()
            if not line:
                continue
            
            idx_match = re.match(r"^\s*(\d+)\.\s*", line)
            original_index = idx_match.group(1) if idx_match else str(i)
            clean_text = INDEX_PATTERN.sub("", line)
            
            if clean_text:
                sentences.append({
                    "original_index": original_index,
                    "text": clean_text
                })
    return sentences

def process_sentence(doc_sentence) -> List[Dict]:
    """Extracts token-level metadata from a parsed Stanza sentence object.

    Args:
        doc_sentence: A Stanza sentence object.

    Returns:
        List[Dict]: List of token dictionaries detailing text, lemma, POS, and dependency data.
    """
    tokens = []
    for word in doc_sentence.words:
        head_text = doc_sentence.words[word.head - 1].text if word.head > 0 else word.text
        tokens.append({
            "text": word.text,
            "lemma": word.lemma if word.lemma else word.text,
            "upos": word.upos,
            "xpos": word.xpos if word.xpos else "",
            "dep": word.deprel,
            "head": head_text,
            "head_index": word.head - 1 if word.head > 0 else word.id - 1,
            "token_index": word.id - 1
        })
    return tokens

def process_batch(sentences: List[str]) -> List[Dict]:
    """Processes a batch of sentence strings through the Stanza pipeline.

    Args:
        sentences (List[str]): List of raw text sentences.

    Returns:
        List[Dict]: Parsed metadata dictionaries per sentence.
    """
    results = []
    try:
        combined_text = "\n\n".join(sentences)
        doc = nlp(combined_text)
        for sent in doc.sentences:
            results.append({
                "sentence": sent.text,
                "tokens": process_sentence(sent)
            })
    except Exception:
        for s in sentences:
            try:
                doc = nlp(s)
                for sent in doc.sentences:
                    results.append({
                        "sentence": sent.text,
                        "tokens": process_sentence(sent)
                    })
            except Exception as e2:
                results.append({
                    "sentence": s,
                    "tokens": [],
                    "error": str(e2)
                })
    return results

## Distributed Batch Processing & Checkpointing

In [ ]:
def load_checkpoint(corpus_name: str) -> Tuple[int, int]:
    """Retrieves the last processed index from the checkpoint file."""
    ckpt_path = CHECKPOINT_DIR / f"checkpoint_{corpus_name}.json"
    if ckpt_path.exists():
        with open(ckpt_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data["last_index"], data["total_processed"]
    return 0, 0

def save_checkpoint(corpus_name: str, last_index: int, total_processed: int):
    """Saves parsing progress to disk to allow for interruption and resumption."""
    ckpt_path = CHECKPOINT_DIR / f"checkpoint_{corpus_name}.json"
    with open(ckpt_path, "w", encoding="utf-8") as f:
        json.dump({
            "last_index": last_index,
            "total_processed": total_processed,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
        }, f)

def process_corpus(corpus_name: str, filepath: Path, batch_size: int = BATCH_SIZE, 
                   checkpoint_every: int = CHECKPOINT_EVERY, max_sentences: Optional[int] = None):
    """Coordinates batch processing of a corpus through Stanza with disk sharding.

    Args:
        corpus_name (str): Identifier for the corpus split (e.g., 'lov', 'ac', 'both').
        filepath (Path): Input file path.
        batch_size (int): Number of sentences per GPU pass.
        checkpoint_every (int): Cadence for writing shards to disk.
        max_sentences (int, optional): Hard limit on processing volume.
    """
    all_sentences = read_corpus_file(filepath, max_lines=max_sentences)
    total = len(all_sentences)
    start_index, total_processed = load_checkpoint(corpus_name)
    
    if start_index >= total:
        return
    
    shard_num = start_index // checkpoint_every
    shard_buffer = []
    t_start = time.time()
    
    pbar = tqdm(range(start_index, total, batch_size), 
                desc=f"Parsing {corpus_name}",
                initial=start_index // batch_size,
                total=(total + batch_size - 1) // batch_size)
    
    for batch_start in pbar:
        batch_end = min(batch_start + batch_size, total)
        batch_sentences = all_sentences[batch_start:batch_end]
        texts = [s["text"] for s in batch_sentences]
        
        parsed = process_batch(texts)
        
        for orig, stanza_result in zip(batch_sentences, parsed):
            record = {
                "corpus": corpus_name,
                "original_index": orig["original_index"],
                "text": orig["text"],
                "stanza_sentence": stanza_result.get("sentence", ""),
                "tokens": stanza_result.get("tokens", []),
            }
            if "error" in stanza_result:
                record["stanza_error"] = stanza_result["error"]
            shard_buffer.append(record)
        
        total_processed = batch_end
        
        if len(shard_buffer) >= checkpoint_every or batch_end >= total:
            shard_path = OUTPUT_DIR / f"{corpus_name}_shard_{shard_num:04d}.jsonl"
            with open(shard_path, "w", encoding="utf-8") as f:
                for record in shard_buffer:
                    f.write(json.dumps(record, ensure_ascii=False) + "\n")
            
            save_checkpoint(corpus_name, batch_end, total_processed)
            shard_buffer = []
            shard_num += 1

def load_all_shards(corpus_name: str) -> List[Dict]:
    """Aggregates processed JSONL shard files back into memory for filtering."""
    shard_files = sorted(OUTPUT_DIR.glob(f"{corpus_name}_shard_*.jsonl"))
    records = []
    for sf in shard_files:
        with open(sf, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    records.append(json.loads(line.strip()))
    return records

## Metadata Scoring & Subpopulation Filtering

In [ ]:
def is_true_participle(token: Dict) -> bool:
    """Validates verb categorization and expected suffix matching."""
    text = token["text"]
    if token["upos"] != "VERB":
        return False
    return text.endswith(LOV_SUFFIX) or text.endswith(AC_SUFFIX)

def get_participle_type(token: Dict) -> str:
    """Categorizes the participle token type based on suffix."""
    if token["text"].endswith(LOV_SUFFIX):
        return "lov"
    elif token["text"].endswith(AC_SUFFIX):
        return "ac"
    return "unknown"

def count_additives(participle_token: Dict, all_tokens: List[Dict]) -> Tuple[int, List[Dict]]:
    """Counts syntactic dependents (additives) modifying the participle."""
    p_idx = participle_token["token_index"]
    additives = []
    for token in all_tokens:
        if token["token_index"] != p_idx and token["head_index"] == p_idx:
            if token["dep"] in ADDITIVE_DEPRELS or token["dep"] in COORD_DEPRELS:
                additives.append(token)
    return len(additives), additives

def find_main_verb(tokens: List[Dict]) -> Optional[Dict]:
    """Identifies the primary root verb of the sequence."""
    for token in tokens:
        if token["dep"] == "root":
            return token
    return None

def classify_position(participle_token: Dict, additives: List[Dict], 
                      main_verb: Optional[Dict], all_tokens: List[Dict]) -> str:
    """Determines the relative syntactic position (PRE/INTRA/POST) of the participle phrase."""
    if main_verb is None:
        return "UNKNOWN"
    
    p_idx = participle_token["token_index"]
    mv_idx = main_verb["token_index"]
    
    phrase_indices = [p_idx] + [a["token_index"] for a in additives]
    phrase_start, phrase_end = min(phrase_indices), max(phrase_indices)
    
    subject = None
    for token in all_tokens:
        if token["head_index"] == mv_idx and token["dep"] in ("nsubj", "nsubj:pass"):
            subject = token
            break
    
    if phrase_end < mv_idx:
        return "INTRA" if subject and phrase_start > subject["token_index"] else "PRE"
    elif phrase_start > mv_idx:
        return "POST"
    else:
        if p_idx < mv_idx:
            return "INTRA" if subject and phrase_start > subject["token_index"] else "PRE"
        return "POST"

def score_sentence(record: Dict) -> Dict:
    """Analyzes Stanza syntax output, scores the sequence quality, and determines retention."""
    tokens = record.get("tokens", [])
    if not tokens or "stanza_error" in record:
        return {**record, "decision": "REJECT_MALFORMED", "quality_score": 0, "participles": []}
    
    word_count = len([t for t in tokens if t["upos"] != "PUNCT"])
    if word_count > 50:
        return {**record, "decision": "REJECT_TOO_COMPLEX", "quality_score": 0, "participles": []}
    if word_count < 3:
        return {**record, "decision": "REJECT_TOO_SHORT", "quality_score": 0, "participles": []}
    
    participles = []
    for token in tokens:
        if is_true_participle(token):
            additive_count, additive_tokens = count_additives(token, tokens)
            main_verb = find_main_verb(tokens)
            position = "UNKNOWN"
            if additive_count > 0 and main_verb:
                position = classify_position(token, additive_tokens, main_verb, tokens)
            
            participles.append({
                "text": token["text"],
                "type": get_participle_type(token),
                "token_index": token["token_index"],
                "additive_count": additive_count,
                "additive_texts": [a["text"] for a in additive_tokens],
                "additive_deprels": [a["dep"] for a in additive_tokens],
                "position": position
            })
    
    if not participles:
        return {**record, "decision": "REJECT_FALSE_PARTICIPLE", "quality_score": 0, "participles": []}
    
    phrases = [p for p in participles if p["additive_count"] > 0]
    if not phrases:
        return {**record, "decision": "REJECT_LONE_PARTICIPLE", "quality_score": 0, "participles": participles}
    
    score = 3
    positions = [p["position"] for p in phrases]
    if all(pos != "UNKNOWN" for pos in positions): score += 2
    if len(phrases) >= 2: score += 2
    types = set(p["type"] for p in participles)
    if "lov" in types and "ac" in types: score += 1
    if "stanza_error" not in record: score += 1
    if 10 <= word_count <= 30: score += 1
    if any(p["position"] == "UNKNOWN" for p in phrases): score -= 2
    if word_count > 40: score -= 2
    
    score = max(0, min(10, score))
    
    position_counts = {}
    for p in phrases:
        position_counts[p["position"]] = position_counts.get(p["position"], 0) + 1
    primary_position = max(position_counts, key=position_counts.get) if position_counts else "UNKNOWN"
    complexity = "multi_phrase" if len(phrases) >= 2 else "single_phrase"
    
    if "lov" in types and "ac" in types: ptype = "both"
    elif "lov" in types: ptype = "lov_only"
    else: ptype = "ac_only"
    
    decision = "KEEP" if score >= 5 else "REJECT_LOW_QUALITY"
    return {
        **record,
        "decision": decision,
        "quality_score": score,
        "participles": participles,
        "sampling_tags": {
            "primary_position": primary_position,
            "participle_type": ptype,
            "complexity": complexity,
            "phrase_count": len(phrases),
            "total_additives": sum(p["additive_count"] for p in phrases),
            "word_count": word_count
        }
    }

## Sampling and Data Export

In [ ]:
def balanced_sample(candidates: List[Dict], target_size: int, 
                    position_ratios: Dict[str, float] = None, seed: int = SEED) -> List[Dict]:
    """Downsamples candidate sequences into a stratified population by structural position."""
    if position_ratios is None:
        position_ratios = {"INTRA": 0.40, "PRE": 0.30, "POST": 0.30}
    random.seed(seed)
    
    quotas = {pos: int(target_size * ratio) for pos, ratio in position_ratios.items()}
    quotas["INTRA"] += target_size - sum(quotas.values())
    
    by_position = defaultdict(list)
    for c in candidates:
        by_position[c["sampling_tags"]["primary_position"]].append(c)
    
    selected = []
    for pos, quota in quotas.items():
        pool = by_position.get(pos, [])
        if len(pool) >= quota:
            selected.extend(pool[:quota])
        else:
            selected.extend(pool)
            
    if len(selected) < target_size:
        selected_ids = set(id(s) for s in selected)
        remaining = [c for c in candidates if id(c) not in selected_ids]
        selected.extend(remaining[:target_size - len(selected)])
        
    return selected[:target_size]

def save_filtered_corpus(corpus: List[Dict], output_path: Path):
    """Persists the sampled corpus to JSON, JSONL, and plain TXT representations."""
    with open(output_path.with_suffix(".json"), "w", encoding="utf-8") as f:
        json.dump(corpus, f, ensure_ascii=False, indent=2)
        
    with open(output_path.with_suffix(".txt"), "w", encoding="utf-8") as f:
        for i, record in enumerate(corpus):
            f.write(f"{i+1}. {record['text']}\n")
            
    with open(output_path.with_suffix(".jsonl"), "w", encoding="utf-8") as f:
        for record in corpus:
            compact = {
                "corpus": record.get("corpus", ""),
                "original_index": record.get("original_index", ""),
                "text": record["text"],
                "quality_score": record.get("quality_score", 0),
                "participles": record.get("participles", []),
                "sampling_tags": record.get("sampling_tags", {})
            }
            f.write(json.dumps(compact, ensure_ascii=False) + "\n")

## Pipeline Execution

In [ ]:
if __name__ == "__main__":
    # 1. Parse Raw Corpora (Outputs to stanza_output/)
    for name, path in CORPUS_FILES.items():
        if path.exists():
            process_corpus(name, path)
            
    # 2. Filter & Subsample Population (Outputs to filtered_corpus/)
    final_combined_corpus = []
    for corpus_name, target in [("lov", TARGET_LOV), ("ac", TARGET_AC), ("both", TARGET_BOTH)]:
        shards = load_all_shards(corpus_name)
        if not shards:
            continue
            
        # Score candidates and preserve top quality
        scored_candidates = []
        for record in shards:
            res = score_sentence(record)
            if res["decision"] == "KEEP":
                scored_candidates.append(res)
        scored_candidates.sort(key=lambda x: x["quality_score"], reverse=True)
        
        # Apply structural stratification limits
        balanced_subset = balanced_sample(scored_candidates, target)
        save_filtered_corpus(balanced_subset, FILTERED_DIR / f"filtered_{target//1000}k_{corpus_name}")
        final_combined_corpus.extend(balanced_subset)

    if final_combined_corpus:
        random.shuffle(final_combined_corpus)
        save_filtered_corpus(final_combined_corpus, FILTERED_DIR / "filtered_120k_combined")